[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_05/MFM_ch5_quasi_score_driven/MFM_ch5_quasi_score_driven.ipynb)

# MFM_ch5_quasi_score_driven

Lecture case study (Blasques, Francq and Laurent, 2023, Journal of Econometrics): GARCH(1,1)-t, Beta-t-GARCH and quasi score-driven QSD GARCH(1,1)-t with an AR(1) mean and the squared VIX (daily horizon) in the variance equation, estimated by maximum likelihood on 11 S&P 500 stocks, 3 January 1995 (or later) to 28 February 2019; likelihood ratio tests of zeta = 0 and xi = zeta, the gap between the tail parameter xi and the downweighting parameter zeta per stock, and news impact curves at the 5, 50 and 95% quantiles of the estimates.

*Modelling Financial Markets (MFM), Chapter 5: Conditional Volatility, GARCH Models. Author: Daniel Traian Pele.*

In [ ]:
# the arch package (Kevin Sheppard) is not preinstalled on Colab
try:
    import arch
except ImportError:
    import subprocess
    import sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'arch'], check=True)

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats, optimize
from scipy.signal import lfilter
from scipy.special import gammaln
from arch import arch_model
from arch.univariate import ARCHInMean, ConstantMean, GARCH, FIGARCH, StudentsT, ZeroMean
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
# font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 11.5
plt.rcParams['ytick.labelsize'] = 11.5
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.4
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 11

# Chart colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Navy     = '#1F2A44'   # reference lines (no grey in charts)
BandBlue = '#C5D2E8'   # light MainBlue tint for confidence / reference bands
Gray, LightGray = Navy, BandBlue   # legacy names kept for importing scripts
Teal     = '#17A2B8'
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'btc': Amber, 'eurron': Forest, 'gold': Purple}

SEED = 42
TABLE_DIR = '.'
NUM = {}


def save_fig(name):
    """Save the figure as transparent PDF and PNG in the current folder."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def acf(x, nlags):
    x = np.asarray(x) - np.mean(x)
    d = np.sum(x ** 2)
    return np.array([np.sum(x[k:] * x[:-k]) / d for k in range(1, nlags + 1)])

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'

# name -> (symbol, label, group, start date)
MARKETS = {
    'sp500':  ('GSPC.INDX',      'S&P 500',                 'Equity', '2000-01-01'),
    'bet':    ('BET',            'BET',                     'Equity', '2000-01-01'),
    'bettr':  ('BETTR.INDX',     'BET-TR',                  'Equity', '2014-09-23'),
    'btc':    ('BTC-USD.CC',     'Bitcoin',                 'Crypto', '2014-09-17'),
    'eurron': ('REF:EUR',        'EUR/RON (BNR reference)', 'FX',     '2005-07-01'),
    'gold':   ('XAUUSD.FOREX',   'Gold (XAU/USD)',          'Commodity', '2000-01-01'),
}
LABELS = {k: v[1] for k, v in MARKETS.items()}

_CACHE = {}


def read_market(symbol):
    """Daily price table of one symbol (local copy of the course data, or GitHub)."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official RON reference rate (annual BNR XML archives)."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END):
    """Daily closing price, cleaned with the chapter conventions."""
    symbol, _, group, start0 = MARKETS[name]
    start = start or start0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    s = read_market(symbol)['close'].loc[start:end]
    s = s[s > 0].dropna()
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    return s.rename(name)


def pct_returns(name, start=None, end=END):
    """Daily log returns in percent, on the series' own calendar."""
    return (100 * np.log(load_close(name, start, end)).diff().dropna()).rename(name)


def periods_per_year(r):
    """Average number of observations per calendar year (actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def load_vix(start='2000-01-01', end=END):
    """The VIX index (30-day implied volatility of the S&P 500, annualised percent)."""
    s = read_market('VIX.INDX')['close'].loc[start:end]
    return s[s.index.dayofweek < 5].rename('vix')


rets = {k: pct_returns(k) for k in MARKETS}
PPY = {k: periods_per_year(rets[k]) for k in MARKETS}
ORDER = list(MARKETS)
MAIN = ['sp500', 'bet', 'btc', 'eurron', 'gold']
for k in ORDER:
    print(f'{LABELS[k]:26s}', rets[k].index[0].date(), '->', rets[k].index[-1].date(), len(rets[k]), 'obs',
          round(PPY[k], 1), 'obs/year')

## GARCH tools

In [ ]:
VOL_SPEC = {'GARCH': dict(vol='GARCH', p=1, o=0, q=1),
            'GJR': dict(vol='GARCH', p=1, o=1, q=1),
            'EGARCH': dict(vol='EGARCH', p=1, o=1, q=1)}
DISTS = {'normal': 'Normal', 't': 'Student-t', 'skewt': 'Skewed-t', 'ged': 'GED'}


class Fit:
    """Result of one fit, in the units of the original series (returns in %)."""

    def __init__(self, res, c, r, vol):
        self.res, self.c, self.r, self.vol = res, c, r, vol
        p = res.params.copy()
        p['mu'] = p['mu'] / c
        if vol == 'EGARCH':
            p['omega'] = p['omega'] - (1 - p['beta[1]']) * np.log(c ** 2)
        else:
            p['omega'] = p['omega'] / c ** 2
        self.params = p
        # robust (Bollerslev-Wooldridge) covariance mapped to the original units with the Jacobian
        # of the transformation: mu/c, omega/c^2 (GARCH, GJR) or omega - (1 - beta) ln c^2 (EGARCH)
        names = list(res.params.index)
        J = np.eye(len(names))
        J[names.index('mu'), names.index('mu')] = 1 / c
        if vol == 'EGARCH':
            J[names.index('omega'), names.index('beta[1]')] = np.log(c ** 2)
        else:
            J[names.index('omega'), names.index('omega')] = 1 / c ** 2
        self.cov = pd.DataFrame(J @ res.param_cov.values @ J.T, index=names, columns=names)
        self.se = pd.Series(np.sqrt(np.diag(self.cov.values)), index=names)
        self.tvalues = p[names] / self.se
        self.pvalues = pd.Series(2 * stats.norm.sf(np.abs(self.tvalues.values)), index=names)
        n = res.nobs
        self.nobs = n
        self.loglik = res.loglikelihood + n * np.log(c)
        k = len(p)
        self.aic = -2 * self.loglik + 2 * k
        self.bic = -2 * self.loglik + k * np.log(n)
        self.sigma = res.conditional_volatility / c
        self.z = res.std_resid
        self.eps = res.resid / c                   # unstandardised residuals, in the units of the original series
        self.converged = res.convergence_flag == 0

    @property
    def persistence(self):
        p = self.params
        if self.vol == 'EGARCH':
            return p['beta[1]']
        g = p.get('gamma[1]', 0.0)
        return p['alpha[1]'] + p['beta[1]'] + g * self.neg_share()

    def neg_share(self):
        """E[z^2 1(z < 0)] for the innovation distribution (0.5 if symmetric); enters the GJR persistence."""
        d = self.res.model.distribution
        if d.name.startswith('Standardized Skew'):
            from scipy import integrate
            par = self.res.params[d.parameter_names()].values
            pdf = lambda x: float(np.exp(d.loglikelihood(par, np.array([x]), np.array([1.0]), individual=True))[0])
            return float(integrate.quad(lambda x: x ** 2 * pdf(x), -np.inf, 0, limit=200)[0])
        return 0.5

    def uncond_var(self):
        p = self.params
        if self.vol == 'EGARCH':
            return np.nan
        return p['omega'] / (1 - self.persistence)

    def forecast_var(self, horizon):
        """Daily variance forecasts for t+1, ..., t+h, in %^2."""
        f = self.res.forecast(horizon=horizon, reindex=False, method='analytic' if self.vol != 'EGARCH' else 'simulation',
                              simulations=2000)
        return f.variance.iloc[-1].values / self.c ** 2


def scale_for(r):
    """Rescaling factor (a power of 10) that makes the standard deviation of order 1."""
    s = r.std()
    return 10.0 if s < 0.5 else 1.0


def fit_garch(r, vol='GARCH', dist='t', mean='Constant', last_obs=None, c=None):
    c = scale_for(r) if c is None else c
    am = arch_model(c * r, mean=mean, dist=dist, **VOL_SPEC[vol])
    res = am.fit(disp='off', last_obs=last_obs, options={'maxiter': 1000})
    return Fit(res, c, r, vol)


def half_life(persistence):
    """Number of days after which half of a variance shock has gone."""
    return np.log(0.5) / np.log(persistence) if 0 < persistence < 1 else np.inf


def news_impact(fit, eps, sigma2_bar=None):
    """sigma^2_t as a function of the shock eps_{t-1}, with sigma^2_{t-1} fixed at the sample variance."""
    p = fit.params
    s2 = fit.r.var() if sigma2_bar is None else sigma2_bar
    if fit.vol == 'EGARCH':
        z = eps / np.sqrt(s2)
        return np.exp(p['omega'] + p['alpha[1]'] * (np.abs(z) - np.sqrt(2 / np.pi)) + p['gamma[1]'] * z
                      + p['beta[1]'] * np.log(s2))
    g = p.get('gamma[1]', 0.0)
    return p['omega'] + (p['alpha[1]'] + g * (eps < 0)) * eps ** 2 + p['beta[1]'] * s2


def sign_bias_test(z, eps):
    """Engle-Ng (1993, eq. 18): z_t^2 on S-_{t-1}, S-_{t-1} eps_{t-1}, S+_{t-1} eps_{t-1}, with eps_{t-1} the
    UNSTANDARDISED residual and z_t the standardised residual; t statistics and the joint test T R^2 ~ chi2(3)."""
    d = pd.concat([pd.Series(z), pd.Series(eps)], axis=1, keys=['z', 'e']).dropna()
    z, e = d['z'].values, d['e'].values
    y = z[1:] ** 2
    el = e[:-1]
    sneg = (el < 0).astype(float)
    X = np.column_stack([np.ones_like(el), sneg, sneg * el, (1 - sneg) * el])
    ols = sm.OLS(y, X).fit()
    joint = len(y) * ols.rsquared
    return {'sign_t': ols.tvalues[1], 'neg_size_t': ols.tvalues[2], 'pos_size_t': ols.tvalues[3],
            'joint': joint, 'joint_p': 1 - stats.chi2.cdf(joint, 3)}


def ljung_box(x, lags=10):
    from statsmodels.stats.diagnostic import acorr_ljungbox
    lb = acorr_ljungbox(pd.Series(np.asarray(x)).dropna(), lags=[lags])
    return float(lb['lb_stat'].iloc[0]), float(lb['lb_pvalue'].iloc[0])


def arch_lm(x, lags=5):
    from statsmodels.stats.diagnostic import het_arch
    lm, lmp, _, _ = het_arch(pd.Series(np.asarray(x)).dropna(), nlags=lags)
    return float(lm), float(lmp)


def ewma_variance(r, lam=0.94, init=250):
    """EWMA forecast for day t+1 (aligned at t+1): s2_{t+1} = lam s2_t + (1 - lam) r_t^2."""
    x = r.values
    s2 = np.empty(len(x) + 1)
    s2[0] = np.mean(x[:init] ** 2)
    for t in range(len(x)):
        s2[t + 1] = lam * s2[t] + (1 - lam) * x[t] ** 2
    return pd.Series(s2[1:-1], index=r.index[1:])


def qlike(proxy, h):
    """QLIKE loss (Patton, 2011), the form used for comparisons: proxy/h + ln h."""
    return proxy / h + np.log(h)


def mz_regression(proxy, h, lags=5):
    """Mincer-Zarnowitz: proxy = a + b h + u, HAC errors; Wald test of a = 0, b = 1."""
    X = sm.add_constant(np.asarray(h))
    ols = sm.OLS(np.asarray(proxy), X).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    w = ols.wald_test((np.eye(2), np.array([0.0, 1.0])), scalar=True)
    return {'a': ols.params[0], 'b': ols.params[1], 'a_se': ols.bse[0], 'b_se': ols.bse[1],
            'r2': ols.rsquared, 'wald': float(w.statistic), 'wald_p': float(w.pvalue)}


def dm_test(loss_a, loss_b, lags=5):
    """Diebold-Mariano: d = L_a - L_b; HAC t statistic for the mean of d (negative: A better)."""
    d = np.asarray(loss_a) - np.asarray(loss_b)
    ols = sm.OLS(d, np.ones_like(d)).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    t = float(ols.tvalues[0])
    return {'mean_diff': float(d.mean()), 't': t, 'p': 2 * (1 - stats.norm.cdf(abs(t)))}

## Helper functions

In [ ]:
QSD_STOCKS = {'AAPL': 'Apple', 'AMZN': 'Amazon', 'BAC': 'Bank of America', 'C': 'Citigroup', 'CSCO': 'Cisco', 'GS': 'Goldman Sachs', 'JPM': 'JPMorgan', 'MSFT': 'Microsoft', 'MS': 'Morgan Stanley', 'NVDA': 'Nvidia', 'WFC': 'Wells Fargo'}

QSD_START, QSD_END = '1995-01-03', '2019-02-28'

QSD_C = 1000.0

QSD = {}

## Analysis

In [ ]:
def qsd_data(ticker):
    """Randamente log (%) ale actiunii si X_t = VIX_t^2/252, pe zilele comune (join pe preturi)."""
    p = read_market(f'{ticker}.US')['adjusted_close'].rename('p')
    v = read_market('VIX.INDX')['close'].rename('vix')
    df = pd.concat([p, v], axis=1, join='inner').loc[:QSD_END].dropna()
    df = df[(df.index.dayofweek < 5) & (df.p > 0)]
    before = df.index[df.index < QSD_START]
    if len(before):
        df = df.loc[before[-1]:]                                # ultimul pret inainte de 3 ianuarie 1995
    r = 100 * np.log(df.p).diff()
    x = df.vix ** 2 / 252
    out = pd.DataFrame({'r': r, 'x': x}).iloc[1:]
    return out


def psi(x, c=QSD_C):
    """Ec. (32): Psi(x) = x (1 - e^{-cx}) / (1 + e^{-cx}) = x tanh(cx/2), aproximare neteda a lui |x|."""
    return x * np.tanh(c * x / 2)


def qsd_filter(theta, r, x):
    """Medie AR(1) si recursia (31); intoarce reziduurile y_t, f_t si eps_t."""
    mu, phi, om, vp, al, be, xi, ze = theta
    y = r[1:] - mu - phi * r[:-1]
    xx = x[1:]                                                  # X_t aliniat cu y_t: f_{t+1} foloseste X_t
    n = len(y)
    f = np.empty(n)
    f[0] = np.var(y)
    for t in range(n - 1):
        e2 = y[t] * y[t] / f[t]
        w = psi((1 + ze) / (1 - 2 * ze + ze * e2))
        f[t + 1] = om + vp * xx[t] + al * w * e2 * f[t] + be * f[t]
    return y, f, y / np.sqrt(f)


def qsd_negll(theta, r, x):
    """Minus log-verosimilitatea: Student-t standardizat cu nu = 1/xi grade de libertate."""
    xi = theta[6]
    if not (0 < xi < 0.5) or not (-1 < theta[7] < 0.5) or theta[2] <= 0 or min(theta[3:6]) < 0:
        return 1e10
    y, f, e = qsd_filter(theta, r, x)
    if not np.all(np.isfinite(f)) or np.any(f <= 0):
        return 1e10
    nu = 1 / xi
    ll = (gammaln((nu + 1) / 2) - gammaln(nu / 2) - 0.5 * np.log(np.pi * (nu - 2)) - 0.5 * np.log(f)
          - (nu + 1) / 2 * np.log1p(e ** 2 / (nu - 2)))
    return -ll.sum()


def fit_qsd(r, x, model, start=None):
    """model: 'GARCH-t' (zeta = 0), 'Beta-t-GARCH' (zeta = xi) sau 'QSD' (zeta liber)."""
    def full(p):
        if model == 'GARCH-t':
            return np.r_[p, 0.0]
        if model == 'Beta-t-GARCH':
            return np.r_[p, p[6]]
        return p
    bnd = [(-1, 1), (-0.5, 0.5), (1e-6, None), (0, None), (0, 1), (0, 1), (0.01, 0.49)]
    if model == 'QSD':
        bnd = bnd + [(-0.99, 0.49)]
    starts = start if start is not None else [np.array([0.05, 0.0, 0.05, 0.1, 0.06, 0.85, 0.15, 0.1])]
    best = None
    for s0 in starts:
        s0 = np.asarray(s0, float)[:len(bnd)]
        res = optimize.minimize(lambda p: qsd_negll(full(p), r, x), s0, method='L-BFGS-B', bounds=bnd)
        res = optimize.minimize(lambda p: qsd_negll(full(p), r, x), res.x, method='Nelder-Mead',
                                options={'maxiter': 4000, 'xatol': 1e-7, 'fatol': 1e-7})
        if best is None or res.fun < best.fun:
            best = res
    return full(best.x), -best.fun


def qsd_se(theta, r, x, h=1e-4):
    """Erori standard din inversa hessianei numerice (modelul QSD nerestrictionat)."""
    k = len(theta)
    H = np.zeros((k, k))
    step = h * np.maximum(np.abs(theta), 1e-2)
    for i in range(k):
        for j in range(i, k):
            ei, ej = np.eye(k)[i] * step[i], np.eye(k)[j] * step[j]
            H[i, j] = H[j, i] = (qsd_negll(theta + ei + ej, r, x) - qsd_negll(theta + ei - ej, r, x)
                                 - qsd_negll(theta - ei + ej, r, x) + qsd_negll(theta - ei - ej, r, x)) / (4 * step[i] * step[j])
    return np.sqrt(np.diag(np.linalg.inv(H)))


def qsd_estimates():
    """Cele trei modele pe fiecare actiune; testele LR ca in Tabelele 4--5 din lucrare."""
    rows = []
    for tk, name in QSD_STOCKS.items():
        d = qsd_data(tk)
        r, x = d.r.values, d.x.values
        g, llg = fit_qsd(r, x, 'GARCH-t')
        b, llb = fit_qsd(r, x, 'Beta-t-GARCH', start=[g[:7]])
        q, llq = fit_qsd(r, x, 'QSD', start=[np.r_[b[:7], b[6]]] + [np.r_[g[:7], z] for z in (0.01, 0.03, 0.06, 0.1)])
        # modelele restrictionate reestimate si din solutia QSD (maximul global al fiecarei verosimilitati)
        g2, llg2 = fit_qsd(r, x, 'GARCH-t', start=[q[:7]])
        b2, llb2 = fit_qsd(r, x, 'Beta-t-GARCH', start=[q[:7]])
        if llg2 > llg:
            g, llg = g2, llg2
        if llb2 > llb:
            b, llb = b2, llb2
        se = qsd_se(q, r, x)
        lr0, lrx = 2 * (llq - llg), 2 * (llq - llb)
        rows.append(dict(ticker=tk, name=name, first=str(d.index[0].date()), last=str(d.index[-1].date()),
                         T=len(d) - 1, xi_garch=g[6], xi_beta=b[6], xi=q[6], zeta=q[7], se_xi=se[6], se_zeta=se[7],
                         alpha=q[4], beta=q[5], varpi=q[3], ll_garch=llg, ll_beta=llb, ll_qsd=llq,
                         lr_zeta0=lr0, p_zeta0=stats.chi2.sf(max(lr0, 0), 1),
                         lr_xi_zeta=lrx, p_xi_zeta=stats.chi2.sf(max(lrx, 0), 1)))
        print(tk, round(q[6], 3), round(q[7], 3), round(lr0, 2), round(lrx, 2))
    t = pd.DataFrame(rows).set_index('ticker')
    t.to_csv(os.path.join(TABLE_DIR, 'ch5_case_qsd.csv'))
    QSD.update(dict(n=len(t), rej_zeta0=int((t.p_zeta0 < 0.05).sum()), rej_xi_zeta=int((t.p_xi_zeta < 0.05).sum()),
                    n_zeta_pos=int((t.zeta > 0).sum()), n_xi_gt_zeta=int((t.xi > t.zeta).sum()),
                    n_sig_xi_gt_zeta=int(((t.p_xi_zeta < 0.05) & (t.xi > t.zeta)).sum()),
                    med_inv_zeta=float(np.median(1 / t.zeta[t.zeta > 0])) if (t.zeta > 0).any() else float('nan'),
                    med_inv_xi_beta=float(np.median(1 / t.xi_beta)),
                    med_zeta=float(t.zeta.median()), med_xi=float(t.xi.median()), med_xi_beta=float(t.xi_beta.median()),
                    min_T=int(t['T'].min()), max_T=int(t['T'].max())))
    with open(os.path.join(TABLE_DIR, 'ch5_case_qsd.json'), 'w') as fh:
        json.dump(QSD, fh, indent=1)
    return t


def fig_qsd_gap(t):
    """xi - zeta pentru fiecare actiune (ca Fig. 2 din lucrare): plin = LR(xi = zeta) semnificativ la 5%."""
    fig, ax = plt.subplots(figsize=(8, 3.5))
    t = t.sort_index()
    pos = np.arange(len(t))
    gap = t.xi - t.zeta
    sig = t.p_xi_zeta < 0.05
    ax.axhline(0, color='#1F2A44', lw=0.6)
    ax.vlines(pos, 0, gap, color=MainBlue, lw=1.0)
    ax.scatter(pos[sig], gap[sig], s=46, color=MainBlue, zorder=3,
               label=r'$\hat\xi - \hat\zeta$, LR test of $\xi = \zeta$ significant at 5%')
    ax.scatter(pos[~sig], gap[~sig], s=46, facecolor='none', edgecolor=MainBlue, lw=1.2, zorder=3,
               label=r'$\hat\xi - \hat\zeta$, not significant')
    ax.scatter(pos + 0.2, t.zeta, s=30, marker='D', color=IDAred, zorder=3,
               label=r'$\hat\zeta$ (downweighting; 0 = GARCH-t)')
    ax.set_xticks(pos)
    ax.set_xticklabels([f'{n} ({k})' for k, n in zip(t.index, t.name)], fontsize=7.5, rotation=30, ha='right')
    ax.set_ylabel(r'QSD GARCH(1,1)-t estimate')
    ax.set_title(f'Quasi score-driven GARCH(1,1)-t, 11 S&P 500 stocks, {QSD_START[:4]}\u2013{QSD_END[:4]}: '
                 '\n' r'tail parameter $\hat\xi$ vs downweighting $\hat\zeta$', color='black')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.36), ncol=3, frameon=False)
    save_fig('ch5_qsd_gap')


def fig_qsd_nic(t):
    """Curbe de impact al stirilor eps -> Psi((1+z)/(1-2z+z eps^2)) eps^2 (ca Fig. 3 din lucrare),
    la cuantilele 5, 50 si 95% ale estimarilor pe cele 11 actiuni."""
    e = np.linspace(-10, 10, 801)
    nic = lambda z: psi((1 + z) / (1 - 2 * z + z * e ** 2)) * e ** 2   # noqa: E731
    fig, ax = plt.subplots(figsize=(7.5, 3.4))
    ax.plot(e, e ** 2, color=Forest, lw=1.6, ls='--', label=r'GARCH(1,1)-t ($\zeta = 0$)')
    qs = [0.05, 0.5, 0.95]
    for m, col, c, ls, name in [('beta', 'xi_beta', IDAred, ':', r'Beta-t-GARCH, $\zeta = \hat\xi$'),
                                ('qsd', 'zeta', MainBlue, '-', r'QSD GARCH(1,1)-t, $\hat\zeta$')]:
        z5, z50, z95 = np.quantile(t[col], qs)
        ax.plot(e, nic(z50), color=c, ls=ls, lw=2.2, label=fr'{name}, median over stocks ({z50:.3f})')
        ax.plot(e, nic(z5), color=c, ls=ls, lw=0.9, label=fr'{name}, 5% and 95% quantiles ({z5:.3f}, {z95:.3f})')
        ax.plot(e, nic(z95), color=c, ls=ls, lw=0.9, label='_nolegend_')
    ax.set_ylim(0, 40)
    ax.set_xlabel(r'Standardised shock $\epsilon_t$')
    ax.set_ylabel(r'Impact on $f_{t+1}$ (units of $\alpha f_t$)')
    ax.set_title('News impact curves, 11 S&P 500 stocks (quantiles over the stocks)', color='black')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=2, frameon=False)
    save_fig('ch5_qsd_nic')
    QSD['nic'] = {f'{m}_{int(qq * 100)}_{s}': float(nic(z)[np.argmin(np.abs(e - s))])
                  for m, col in [('beta', 'xi_beta'), ('qsd', 'zeta')] for qq in qs
                  for z in [np.quantile(t[col], qq)] for s in [3, 5, 10]}
    with open(os.path.join(TABLE_DIR, 'ch5_case_qsd.json'), 'w') as fh:
        json.dump(QSD, fh, indent=1)

## Run

In [ ]:
tab = qsd_estimates()
print(tab.round(4).T)
fig_qsd_gap(tab)
fig_qsd_nic(tab)
print(json.dumps(QSD, indent=1))

![ch5_qsd_gap](./ch5_qsd_gap.png)

Output: `ch5_qsd_gap.pdf`

![ch5_qsd_nic](./ch5_qsd_nic.png)

Output: `ch5_qsd_nic.pdf`